In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver - Data Preparation

This notebook prepares the dataset for transformer-based ranking models.

Goals:

- Convert MCQ format into Question-Option pairs
- Create binary labels
- Prepare training and validation splits
- Analyze ranking dataset characteristics

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_colwidth", 500)

In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [7]:
def create_ranking_dataset(df):

    rows = []

    for _, row in df.iterrows():

        question = row["prompt"]
        answer = row["answer"]

        for option in ["A","B","C","D","E"]:

            rows.append({
                "id": row["id"],
                "question": question,
                "option_label": option,
                "option_text": row[option],
                "label": int(option == answer)
            })

    return pd.DataFrame(rows)

In [8]:
ranking_train = create_ranking_dataset(train)

ranking_train.shape

(10000, 5)

In [9]:
ranking_train.head(10)

,id,question,option_label,option_text,label
0,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,A,"Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.",0
1,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,B,"Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.",1
2,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,C,"Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.",0
3,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,D,Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present are interconnected and the future is predetermined. Human beings do not have free will.,0
4,1,Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.,E,"Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneously. Humans exist outside of this illusion and are guided by a higher power.",0
5,2,What is accelerator-based light-ion fusion?,A,"Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.",1
6,2,What is accelerator-based light-ion fusion?,B,"Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.",0
7,2,What is accelerator-based light-ion fusion?,C,"Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.",0
8,2,What is accelerator-based light-ion fusion?,D,"Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 100 kV between the electrodes.",0
9,2,What is accelerator-based light-ion fusion?,E,"Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fission reactions. This method is relatively 

In [10]:
ranking_train["label"].value_counts()

label
0    8000
1    2000
Name: count, dtype: int64

In [11]:
ranking_train["text"] = (
    "Question: " +
    ranking_train["question"] +
    " [SEP] Option: " +
    ranking_train["option_text"]
)

In [12]:
ranking_train[[
    "text",
    "label"
]].head()

,text,label
0,"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Option: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.",0
1,"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Option: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.",1
2,"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Option: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present.",0
3,Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Option: Martin Heidegger believes that the relationship between time and human existence is cyclical. The past and present are interconnected and the future is predetermined. Human beings do not have free will.,0
4,"Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Option: Martin Heidegger believes that time is an illusion, and the past, present, and future are all happening simultaneously. Humans exist outside of this illusion and are guided by a higher power.",0


In [13]:
ranking_train["text_length"] = (
    ranking_train["text"]
    .str.len()
)

In [14]:
ranking_train["text_length"].describe()

count    10000.000000
mean       307.615200
std        116.763815
min         86.000000
25%        220.750000
50%        285.000000
75%        376.000000
max        856.000000
Name: text_length, dtype: float64

In [15]:
question_groups = ranking_train["question"]

In [16]:
print(
    "Unique Questions:",
    ranking_train["question"].nunique()
)

Unique Questions: 1758


In [17]:
ranking_train["fold"] = -1

In [18]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (_, val_idx) in enumerate(
    skf.split(
        ranking_train,
        ranking_train["label"]
    )
):

    ranking_train.loc[
        val_idx,
        "fold"
    ] = fold

In [19]:
ranking_train["fold"].value_counts()

fold
1    2000
3    2000
2    2000
0    2000
4    2000
Name: count, dtype: int64

In [20]:
ranking_train.to_csv(
    "/kaggle/working/ranking_train.csv",
    index=False
)

print("Saved")

Saved


In [21]:
ranking_train.shape

(10000, 8)

In [22]:
ranking_train["label"].value_counts()

label
0    8000
1    2000
Name: count, dtype: int64

In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base"
)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [24]:
token_lengths = ranking_train["text"].apply(
    lambda x: len(
        tokenizer.encode(
            x,
            add_special_tokens=True
        )
    )
)

token_lengths.describe()

count    10000.000000
mean        59.083400
std         22.162337
min         17.000000
25%         43.000000
50%         55.000000
75%         72.000000
max        193.000000
Name: text, dtype: float64